In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import xgboost as xgb

# =========================
# 0) Helper: remove outliers
# =========================
def remove_outliers_iqr(df, col, k=1.5):
    """
    Remove outliers using IQR rule.
    Keep rows in [Q1 - k*IQR, Q3 + k*IQR].
    """
    x = df[col].astype(float)
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    mask = (x >= low) & (x <= high)
    return df.loc[mask].copy(), (low, high)

# Load data
df = pd.read_csv('/content/data-train-ml_house_prices.csv')

# Chọn features và target
features = ['street', 'ward', 'district', 'bedroom', 'bathroom', 'area(m2)']
target = 'price(ty-VND)'

# =========================
# 1) Remove outliers (target)
# =========================
USE_OUTLIER_FILTER = True
IQR_K = 1.5  # thử 1.5 (chuẩn), nếu muốn mạnh tay: 1.0; nhẹ tay: 2.0~3.0

if USE_OUTLIER_FILTER:
    before = len(df)
    df, (low, high) = remove_outliers_iqr(df, target, k=IQR_K)
    after = len(df)
    print(f"[Outlier filter] {target}: keep in [{low:.3f}, {high:.3f}]")
    print(f"[Outlier filter] removed {before - after} rows ({(before-after)/before*100:.2f}%)")

# =========================
# 2) Prepare X, y
# =========================
X = df[features].copy()
y = df[target].copy()

# Xử lý categorical features (street và district)
label_encoder_street = LabelEncoder()
label_encoder_district = LabelEncoder()

X.loc[:, 'street'] = label_encoder_street.fit_transform(X['street'].astype(str))
X.loc[:, 'district'] = label_encoder_district.fit_transform(X['district'].astype(str))

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train XGBoost
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# =========================
# 3) Metrics
# =========================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

eps = 1e-9
mape = np.mean(np.abs((y_test - y_pred) / (np.abs(y_test) + eps))) * 100

print("===== EVALUATION METRICS =====")
print(f"MAE  : {mae:.6f}")
print(f"MSE  : {mse:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R2   : {r2:.6f}")
print(f"MAPE : {mape:.2f}%")

# =========================
# 4) Cross-validation (R2)
# =========================
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2_scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)

print("\n===== CROSS-VALIDATION (R2) =====")
print("R2 per fold:", np.round(cv_r2_scores, 6))
print(f"Mean R2    : {cv_r2_scores.mean():.6f}")
print(f"Std R2     : {cv_r2_scores.std():.6f}")

# =========================
# 5) Feature importance
# =========================
importances = model.feature_importances_
fi = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

print("\n===== FEATURE IMPORTANCE =====")
print(fi.to_string(index=False))

# =========================
# 6) Sample predictions
# =========================
compare = X_test.copy()
compare['y_true'] = y_test.values
compare['y_pred'] = y_pred
compare['abs_error'] = np.abs(compare['y_true'] - compare['y_pred'])
compare['pct_error_%'] = (compare['abs_error'] / (np.abs(compare['y_true']) + eps)) * 100

print("\n===== SAMPLE PREDICTIONS (10 rows) =====")
print(compare[['y_true', 'y_pred', 'abs_error', 'pct_error_%']].head(10).to_string(index=False))

# =========================
# 7) Worst errors
# =========================
worst = compare.sort_values('abs_error', ascending=False).head(10)
print("\n===== WORST 10 ERRORS =====")
print(worst[['y_true', 'y_pred', 'abs_error', 'pct_error_%', 'street', 'ward', 'district', 'bedroom', 'bathroom', 'area(m2)']].to_string(index=False))


[Outlier filter] price(ty-VND): keep in [-3.375, 16.425]
[Outlier filter] removed 660 rows (8.09%)
===== EVALUATION METRICS =====
MAE  : 1.011042
MSE  : 2.112387
RMSE : 1.453405
R2   : 0.803266
MAPE : 17.00%

===== CROSS-VALIDATION (R2) =====
R2 per fold: [0.798635 0.80945  0.804937 0.791468 0.794408]
Mean R2    : 0.799780
Std R2     : 0.006623

===== FEATURE IMPORTANCE =====
 feature  importance
bathroom    0.291145
district    0.190115
 bedroom    0.186451
area(m2)    0.179062
    ward    0.080897
  street    0.072332

===== SAMPLE PREDICTIONS (10 rows) =====
 y_true    y_pred  abs_error  pct_error_%
   4.50  4.339200   0.160800     3.573343
   8.50  8.116369   0.383631     4.513303
   5.60  6.244757   0.644757    11.513512
   2.53  2.795885   0.265885    10.509301
   7.00  6.927324   0.072676     1.038224
   6.60  8.370084   1.770084    26.819452
   9.10  7.263333   1.836667    20.183150
  12.50 12.352880   0.147120     1.176956
   3.20  3.867612   0.667612    20.862879
   5.60  5.7

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ======================
# 1. Load data
# ======================
df = pd.read_csv('/content/data-train-ml_house_prices-linear.csv')

# ======================
# 2. Define features & target
# ======================
features = ['district', 'bedroom', 'bathroom', 'area(m2)']
target = 'price(ty-VND)'

X = df[features]
y = df[target]

# ======================
# 3. Preprocessing
# ======================
num_features = ['bedroom', 'bathroom', 'area(m2)']
cat_features = ['district']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

# ======================
# 4. Model
# ======================
model = LinearRegression()

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model)
])

# ======================
# 5. Train / Test split
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ======================
# 6. Train
# ======================
pipeline.fit(X_train, y_train)

# ======================
# 7. Evaluate
# ======================
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("===== Evaluation =====")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R2   : {r2:.3f}")

# ======================
# 8. Predict sample
# ======================
sample = pd.DataFrame({
    'district': ['Quan 7'],
    'bedroom': [3],
    'bathroom': [2],
    'area(m2)': [90]
})

price_pred = pipeline.predict(sample)

print("\n===== Sample Prediction =====")
print(f"Predicted price: {price_pred[0]:.2f} ty VND")


===== Evaluation =====
MAE  : 2.775
RMSE : 4.940
R2   : 0.503

===== Sample Prediction =====
Predicted price: 10.11 ty VND
